# Agile Team Simulator
**Carbon vs Silicon vs Mixed teams — Scrum / Kanban — WIP limit sensitivity**

Models throughput as a function of team composition, methodology, and WIP limits.

Key mechanic:
- Carbon teams degrade non-linearly beyond their optimal WIP range;
- Silicon teams hit a hard throughput ceiling instead of degrading.

In [5]:
import math
import random
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

## 1. Core model

### WIP penalty functions

The central mechanic. Each team type has a different response curve to WIP pressure:

- **Carbon**: bell-curve efficiency — optimal at ~1.5× team size, drops off sharply above 3×
- **Silicon**: flat until a hard API/compute ceiling, then hard cap (no cognitive degradation)
- **Mixed**: weighted blend of both curves

In [6]:
def carbon_wip_efficiency(wip: int, team_size: int = 5) -> float:
    """
    Carbon (human) teams: efficiency peaks around 1.5x team_size WIP,
    then degrades due to context-switching and coordination overhead.
    Returns a multiplier in [0.2, 1.0].
    """
    optimal = 1.5 * team_size
    if wip <= 0:
        return 0.0
    if wip <= optimal:
        # Ramp up to optimal (slight underload penalty)
        return 0.7 + 0.3 * (wip / optimal)
    else:
        # Non-linear degradation above optimal
        overload_ratio = (wip - optimal) / optimal
        penalty = 1.0 - 0.55 * (1 - math.exp(-1.8 * overload_ratio))
        return max(0.2, penalty)


def silicon_wip_efficiency(wip: int, team_size: int = 5, ceiling_multiplier: float = 4.0) -> float:
    """
    Silicon (AI agent) teams: near-flat efficiency up to a hard ceiling
    (rate limits, compute constraints). No cognitive degradation.
    Returns a multiplier in [0.0, 1.0].
    """
    ceiling = ceiling_multiplier * team_size
    if wip <= 0:
        return 0.0
    if wip <= ceiling:
        # Very slight overhead for coordination between agents
        return 1.0 - 0.02 * (wip / ceiling)
    else:
        # Hard ceiling — excess WIP just queues, no additional throughput
        return ceiling / wip  # effective throughput normalised


def mixed_wip_efficiency(wip: int, team_size: int = 5,
                         carbon_ratio: float = 0.5) -> float:
    """
    Mixed team: weighted blend. The human fraction limits the ceiling;
    the agent fraction lifts the floor. Sweet spot is 50/50 <-- yes, a guesstimation.
    """
    c = carbon_wip_efficiency(wip, team_size)
    s = silicon_wip_efficiency(wip, team_size)
    # Synergy bonus: agents handle overflow, humans handle ambiguity
    synergy = 0.08 * (1 - carbon_ratio) * carbon_ratio  # peaks at 50/50
    return min(1.0, carbon_ratio * c + (1 - carbon_ratio) * s + synergy)

In [7]:
# --- Plot the efficiency curves ---
# TODO: check that the curves are consistent with the model

wip_range = list(range(1, 31))
team_size = 5

fig = go.Figure()
fig.add_trace(go.Scatter(x=wip_range,
    y=[carbon_wip_efficiency(w, team_size) for w in wip_range],
    name='Carbon', line=dict(color='#3266ad', width=2)))

fig.add_trace(go.Scatter(x=wip_range,
    y=[silicon_wip_efficiency(w, team_size) for w in wip_range],
    name='Silicon', line=dict(color='#1D9E75', width=2, dash='dash')))

fig.add_trace(go.Scatter(x=wip_range,
    y=[mixed_wip_efficiency(w, team_size) for w in wip_range],
    name='Mixed (50/50)', line=dict(color='#BA7517', width=2, dash='dot')))

fig.add_vline(x=1.5 * team_size, line_dash='dot', line_color='gray',
    annotation_text='Carbon optimum', annotation_position='top right')

fig.update_layout(
    title='WIP efficiency curves (team_size=5)',
    xaxis_title='WIP (items in progress)',
    yaxis_title='Efficiency multiplier',
    yaxis=dict(range=[0, 1.1]),
    height=380,
    template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

## 2. Our team and backlog data classes

In [8]:
import dataclasses
from typing import Literal

TeamType = Literal['carbon', 'silicon', 'mixed']
Method = Literal['scrum', 'kanban']


@dataclasses.dataclass
class Team:
    name: str
    team_type: TeamType
    size: int = 5                    # people or agent instances
    wip_limit: int = 8               # explicit WIP limit (Kanban-style)
    carbon_ratio: float = 0.5        # only used for mixed teams

    # Base throughput in items/person/day (before WIP efficiency)
    BASE_RATES = {'carbon': 0.36, 'silicon': 0.64, 'mixed': 0.50}
    # Ceremony overhead fraction of capacity consumed per day
    SCRUM_OVERHEAD = {'carbon': 0.18, 'silicon': 0.04, 'mixed': 0.12}
    KANBAN_OVERHEAD = {'carbon': 0.05, 'silicon': 0.01, 'mixed': 0.04}

    # Retro learning boost per sprint (Scrum only)
    RETRO_BOOST = {'carbon': 0.04, 'silicon': 0.0, 'mixed': 0.02}
    MAX_RETRO_BOOST = {'carbon': 0.30, 'silicon': 0.0, 'mixed': 0.15}

    def __post_init__(self):
        self._retro_accumulated = 0.0

    def apply_retro(self):
        """Call once per sprint end. Accumulates velocity improvement."""
        boost = self.RETRO_BOOST[self.team_type]
        cap = self.MAX_RETRO_BOOST[self.team_type]
        self._retro_accumulated = min(self._retro_accumulated + boost, cap)

    def throughput_today(self, current_wip: int, method: Method) -> float:
        """Items completed today given current WIP and methodology."""
        base = self.BASE_RATES[self.team_type] * self.size
        overhead = (self.SCRUM_OVERHEAD if method == 'scrum'
                    else self.KANBAN_OVERHEAD)[self.team_type]

        if self.team_type == 'carbon':
            wip_eff = carbon_wip_efficiency(current_wip, self.size)
        elif self.team_type == 'silicon':
            wip_eff = silicon_wip_efficiency(current_wip, self.size)
        else:
            wip_eff = mixed_wip_efficiency(current_wip, self.size, self.carbon_ratio)

        return base * wip_eff * (1 - overhead) * (1 + self._retro_accumulated)


@dataclasses.dataclass
class WorkItem:
    item_type: Literal['feature', 'bug', 'story']
    high_priority: bool = False
    complexity: float = 1.0          # effort multiplier (1.0 = 1 story point)
    started_day: int | None = None
    done_day: int | None = None

    @property
    def cycle_time(self) -> int | None:
        if self.started_day is not None and self.done_day is not None:
            return self.done_day - self.started_day
        return None

## 3. Simulation engine

In [9]:
def build_backlog(
    n_features: int = 20,
    n_bugs: int = 15,
    n_stories: int = 25,
    pct_high_priority: float = 0.20,
    seed: int = 42,
) -> list[WorkItem]:
    rng = random.Random(seed)
    items = []
    for _ in range(n_features):
        items.append(WorkItem('feature', complexity=rng.uniform(0.8, 2.0)))
    for _ in range(n_bugs):
        items.append(WorkItem('bug', complexity=rng.uniform(0.3, 1.2)))
    for _ in range(n_stories):
        items.append(WorkItem('story', complexity=rng.uniform(0.5, 1.5)))

    # Mark high-priority items — pull from bugs first, then features
    n_hp = int(len(items) * pct_high_priority)
    bugs = [i for i in items if i.item_type == 'bug']
    rest = [i for i in items if i.item_type != 'bug']
    for item in (bugs + rest)[:n_hp]:
        item.high_priority = True

    # Priority sort: high-priority first, then by complexity (small bugs first)
    items.sort(key=lambda i: (not i.high_priority, i.complexity))
    return items

def run_simulation(
    teams: list[Team],
    backlog: list[WorkItem],
    method: Method = 'scrum',
    sprint_length: int = 10,
    max_days: int = 200,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Runs the simulation and returns a tidy dataframe with one row per (day, team).
    """
    rng = random.Random(seed)
    remaining = [item for item in backlog]  # shallow copy — items are mutable
    in_progress: dict[str, list[WorkItem]] = {t.name: [] for t in teams}
    done_items: list[WorkItem] = []
    records = []

    sprint = 0

    for day in range(1, max_days + 1):
        # --- Sprint boundary (Scrum) ---
        if method == 'scrum' and (day - 1) % sprint_length == 0:
            sprint += 1
            if sprint > 1:
                for team in teams:
                    team.apply_retro()

        # --- Each team pulls and completes work ---
        for team in teams:
            wip = in_progress[team.name]

            # Pull new items up to WIP limit
            while len(wip) < team.wip_limit and remaining:
                item = remaining.pop(0)
                item.started_day = day
                wip.append(item)

            # How much work does this team complete today?
            capacity = team.throughput_today(len(wip), method)

            # Process items (fractional progress tracked via complexity)
            completed_today = 0
            still_wip = []
            for item in wip:
                if capacity >= item.complexity:
                    capacity -= item.complexity
                    item.done_day = day
                    done_items.append(item)
                    completed_today += 1
                else:
                    # Partial progress — reduce complexity for tomorrow
                    item.complexity -= capacity
                    capacity = 0
                    still_wip.append(item)

            in_progress[team.name] = still_wip

            records.append({
                'day': day,
                'sprint': sprint if method == 'scrum' else None,
                'team': team.name,
                'team_type': team.team_type,
                'wip_limit': team.wip_limit,
                'actual_wip': len(wip),
                'completed_today': completed_today,
                'retro_boost': team._retro_accumulated,
                'backlog_remaining': len(remaining),
            })

        if not remaining and all(len(v) == 0 for v in in_progress.values()):
            break  # all done

    df = pd.DataFrame(records)
    df['cumulative_done'] = df.groupby('team')['completed_today'].cumsum()
    df['total_done'] = df.groupby('day')['completed_today'].transform('sum')
    return df